# 🏌️ DOH 스윙분석 — 원클릭 (Colab 무료)

**하는 법:** 위에서 **런타임 ▸ 런타임 유형 변경 ▸ T4 GPU ▸ 저장**, 그다음 **런타임 ▸ 모두 실행**.
맨 아래에 화면이 뜨면 거기서 **영상 올리고 [분석하기]** 만 누르면 됨. (셀 하나씩 안 만져도 됨)

### 1칸 — 준비 (모델 로드)
`>>> NLF OK` 뜨면 성공. 처음 한 번만 오래 걸림.

In [ ]:
# 1칸 · 준비 (1~2분). 끝에 >>> NLF OK 뜨면 성공.
!pip -q install "gradio>=4" 2>/dev/null
import torch, torchvision, torchvision.ops, os, urllib.request
print("torch", torch.__version__, "| GPU", torch.cuda.is_available())
URL = "https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript"
M = "nlf_l_multi_0.3.2.torchscript"
if (not os.path.exists(M)) or os.path.getsize(M) < 10_000_000:
    print("NLF 모델 다운로드(470MB)..."); urllib.request.urlretrieve(URL, M)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.jit.load(M).to(DEVICE).eval()
print(">>> NLF OK  (device =", DEVICE, ")")

### 2칸 — 분석 화면
실행하면 **바로 아래에 업로드 화면**이 떠. 영상 올리고 각도(정면/측면)·주손 고른 뒤 **분석하기**.

In [ ]:
# 2칸 · UI 실행. 실행하면 이 아래에 화면이 뜸 → 영상 올리고 [분석하기].
BR = "claude/ai-video-analysis-engine-wlr06k"
BASE = f"https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/{BR}/pose3d_poc"
import urllib.request, numpy as np, cv2, gradio as gr
for f in ("wham_golf_rotation.py", "wham_golf_metrics.py"):
    urllib.request.urlretrieve(f"{BASE}/{f}", f)
from wham_golf_rotation import build_v1

def video_to_joints(path, max_side=720):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 60.0; J = []
    with torch.inference_mode():
        while True:
            ok, fr = cap.read()
            if not ok: break
            h, w = fr.shape[:2]; sc = max_side / max(h, w)
            if sc < 1: fr = cv2.resize(fr, (int(w*sc), int(h*sc)), interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
            pred = model.detect_smpl_batched(t); per = pred["joints3d"][0]
            if per is None or len(per) == 0: continue
            kp = per[0]; kp = kp.detach().cpu().numpy() if torch.is_tensor(kp) else np.asarray(kp)
            J.append(kp)
    cap.release(); return np.asarray(J), round(float(fps), 2)

META = {
 "VF015":("어깨 회전 @탑","회전"),"VF018":("골반 회전 @탑","회전"),"VF020":("X-Factor @탑","회전"),"VF075":("힙 클리어","회전"),
 "VF002":("척추각 @어드레스","자세 (측면)"),"VF038":("자세 유지 (어드→탑)","자세 (측면)"),
 "VF076":("척추각 유지","자세 (측면)"),"VF022":("어깨 플레인","자세 (측면)"),"VF001":("좌우 틸트","자세 (정면)"),
 "VF011":("리드팔 곧음","팔·무릎"),"VF012":("트레일팔 굽힘","팔·무릎"),"VF027":("트레일 팔꿈치각","팔·무릎"),
 "VF087":("리드팔 굽힘 @임팩트","팔·무릎"),"VF039":("리드무릎 굽힘변화","팔·무릎"),
 "VF040":("트레일무릎 굽힘","팔·무릎"),"VF088":("리드무릎각 @임팩트","팔·무릎"),
 "VF031":("머리 스웨이","스웨이 (정면)"),"VF034":("골반 스웨이","스웨이 (정면)"),
 "VF113":("백스윙 시간","템포"),"VF114":("다운스윙 시간","템포"),"VF111":("템포 비율","템포"),
}
FLAG = {"view_mismatch":"이 각도에선 측정 불가","off_axis_view":"각도 미상·참고용","depth_unreliable":"깊이 불안정",
        "interpolated_event":"이벤트 보간","club_not_detected":"클럽 미검출"}
GROUPS = ["회전","자세 (측면)","자세 (정면)","팔·무릎","스웨이 (정면)","템포"]

def _card(f):
    name = META.get(f["feature_id"], (f.get("name",""),"기타"))[0]
    na = f.get("value") is None
    flags = " · ".join(FLAG.get(x, x) for x in f.get("error_flags", []))
    if na:
        val = f'<div style="font-size:15px;color:#6e7681;font-weight:700">{flags or "측정 불가"}</div>'
        bar = ""
    else:
        u = "°" if f["unit"]=="deg" else ("" if f["unit"]=="ratio" else " "+f["unit"])
        val = f'<div style="font-size:25px;font-weight:800;color:#e6edf3">{f["value"]}<span style="font-size:13px;color:#8b949e">{u}</span></div>'
        cf = int(round(f.get("confidence",0)*100))
        bar = f'<div style="margin-top:8px;height:4px;border-radius:2px;background:#232b36"><div style="width:{cf}%;height:100%;background:#3fb950"></div></div>'
        if flags: bar += f'<div style="margin-top:6px;font-size:11px;color:#d29922">{flags}</div>'
    return (f'<div style="background:#0d1117;border:1px solid #232b36;border-radius:12px;padding:13px">'
            f'<div style="font-size:12px;color:#8b949e;min-height:30px;line-height:1.3">{name}</div>{val}{bar}</div>')

def render(inst):
    ev = {e["p"]: e["frame"] for e in inst.get("swing_events", [])}
    q = inst.get("quality", {}); src = inst.get("source", {})
    byg = {g: [] for g in GROUPS}
    for f in inst.get("features", []):
        g = META.get(f["feature_id"], (None,"기타"))[1]; byg.setdefault(g, []).append(f)
    h = ['<div style="font-family:-apple-system,Segoe UI,Malgun Gothic,sans-serif;background:#161b22;border:1px solid #232b36;border-radius:14px;padding:16px;color:#e6edf3">']
    h.append('<div style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:6px">')
    for lab, key in [("촬영", src.get("camera_view","?")), ("P1", ev.get("P1","?")), ("P4", ev.get("P4","?")), ("P7", ev.get("P7","?")), ("프레임", src.get("duration_frames","?"))]:
        h.append(f'<span style="background:#0d1117;border:1px solid #232b36;border-radius:9px;padding:6px 11px;font-size:12px;color:#8b949e">{lab} <b style="color:#58a6ff">{key}</b></span>')
    h.append("</div>")
    for g in GROUPS:
        fs = byg.get(g)
        if not fs: continue
        h.append(f'<div style="margin-top:16px;font-size:14px;font-weight:700">{g}</div>')
        h.append('<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(145px,1fr));gap:10px;margin-top:8px">')
        h.append("".join(_card(f) for f in fs)); h.append("</div>")
    warns = [w for w in q.get("warnings", []) if not w.startswith("joints=") and not w.startswith("object_engine")]
    h.append(f'<div style="margin-top:14px;font-size:13px;color:#8b949e">종합 신뢰도 <b style="color:#e6edf3">{int(round(q.get("overall_confidence",0)*100))}%</b> · 각도적합 {"✅" if q.get("view_match") else "⚠️"}{(" · "+" · ".join(warns)) if warns else ""}</div>')
    h.append("</div>")
    return "".join(h)

def analyze_fn(video, view, hand):
    if not video: return "<p style='color:#8b949e'>영상을 올려주세요.</p>"
    J, fps = video_to_joints(video)
    if J.ndim != 3 or J.shape[0] < 5:
        return "<p style='color:#f85149'>사람/프레임을 충분히 못 잡았어요. 전신이 크게 나오는 영상으로.</p>"
    inst = build_v1(J, skeleton="smpl", view=view, hand=hand, fps=fps, video_id="swing")
    return render(inst)

with gr.Blocks(title="DOH 스윙분석", theme=gr.themes.Base()) as demo:
    gr.Markdown("## 🏌️ DOH 스윙분석\n영상 올리고 **[분석하기]** → 회전·척추각·자세·팔·무릎·템포. (측면=척추/플레인, 정면=스웨이/좌우틸트)")
    with gr.Row():
        vid = gr.Video(label="스윙 영상 (mp4/mov)", sources=["upload"])
        with gr.Column():
            view = gr.Radio(["FO", "DTL"], value="FO", label="촬영 각도  (FO=정면 / DTL=측면)")
            hand = gr.Radio(["right", "left"], value="right", label="주손")
            btn = gr.Button("분석하기", variant="primary", size="lg")
    out = gr.HTML()
    btn.click(analyze_fn, [vid, view, hand], out)

demo.launch(debug=False)